# Module 01: Lag Polynomial Visualization

## Learning Objectives
By completing this notebook, you will:
1. Build interactive visualizations of MIDAS weight functions
2. Understand how $\theta$ parameters control the weight shape
3. Visualize the relationship between weight parameters and R² surface
4. Identify the optimal weight parameters visually before running NLS

## Prerequisites
- Notebooks 01 and 02 completed

## Estimated Time: 10 minutes

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.rcParams['figure.dpi'] = 110

RESOURCES_DIR = '../../module_00_foundations/resources'
if not os.path.exists(RESOURCES_DIR):
    RESOURCES_DIR = '../../../module_00_foundations/resources'

print("Imports complete.")

## 1. Setup

In [ ]:
# Load data
gdp = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'gdp_quarterly.csv'),
    index_col='date', parse_dates=True
).squeeze()
gdp.index = pd.PeriodIndex(gdp.index, freq='Q')

ip = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'industrial_production_monthly.csv'),
    index_col='date', parse_dates=True
).squeeze()
ip.index = pd.PeriodIndex(ip.index, freq='M')

# MIDAS matrix: 3 quarterly lags = K=9
def build_midas_matrix(y_q, x_m, n_q=3, m=3):
    K = n_q * m
    T = len(y_q)
    X = np.full((T, K), np.nan)
    for t, q in enumerate(y_q.index):
        lm = q.end_time.to_period('M')
        for j in range(K):
            tgt = lm - j
            if tgt in x_m.index:
                X[t, j] = x_m.iloc[x_m.index.get_loc(tgt)]
    Y = y_q.values
    ok = ~(np.isnan(X).any(1) | np.isnan(Y))
    return Y[ok], X[ok], y_q.index[ok]

Y, X, dates = build_midas_matrix(gdp, ip)
K = X.shape[1]
T = len(Y)

def beta_weights(K, t1, t2):
    if t1 <= 0 or t2 <= 0: return np.ones(K)/K
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, t1, t2)
    s = raw.sum()
    return raw/s if s > 1e-12 else np.ones(K)/K

def midas_r2(Y, X, t1, t2):
    """Compute R² for a given theta using OLS-profile method."""
    w = beta_weights(X.shape[1], t1, t2)
    x_w = X @ w
    model = LinearRegression().fit(x_w.reshape(-1,1), Y)
    yh = model.predict(x_w.reshape(-1,1))
    return r2_score(Y, yh)

print(f"Ready: T={T}, K={K}")

## 2. The $R^2$ Surface Over the $(\theta_1, \theta_2)$ Parameter Space

For each combination of $(\theta_1, \theta_2)$, we compute the R² achievable by the corresponding Beta polynomial MIDAS model. This surface shows:
- Where the global optimum lies
- Whether the surface is smooth (easy NLS convergence) or rugged (multiple local optima)
- How sensitive R² is to the weight function shape

In [ ]:
# Grid search over theta parameter space
theta1_grid = np.linspace(0.2, 5.0, 25)
theta2_grid = np.linspace(0.2, 8.0, 30)

R2_surface = np.zeros((len(theta1_grid), len(theta2_grid)))

for i, t1 in enumerate(theta1_grid):
    for j, t2 in enumerate(theta2_grid):
        R2_surface[i, j] = midas_r2(Y, X, t1, t2)

# Find the grid-optimal parameters
best_idx = np.unravel_index(R2_surface.argmax(), R2_surface.shape)
best_t1 = theta1_grid[best_idx[0]]
best_t2 = theta2_grid[best_idx[1]]
best_r2 = R2_surface[best_idx]

print(f"Grid-optimal parameters: θ₁={best_t1:.2f}, θ₂={best_t2:.2f}")
print(f"Grid-optimal R²: {best_r2:.4f}")
print(f"R² at uniform (θ₁=1, θ₂=1): {midas_r2(Y, X, 1.0, 1.0):.4f}")

# Plot the R² surface
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'R² Surface Over Beta Polynomial Parameter Space\n'
             f'(GDP ~ IP, K={K} lags, T={T} quarters)',
             fontsize=12, fontweight='bold')

# Contour map
T1, T2 = np.meshgrid(theta1_grid, theta2_grid, indexing='ij')

ax1 = axes[0]
cf = ax1.contourf(T1, T2, R2_surface, levels=20, cmap='YlOrRd')
ax1.contour(T1, T2, R2_surface, levels=10, colors='white', alpha=0.4, linewidths=0.5)
plt.colorbar(cf, ax=ax1, label='R²')
ax1.scatter(best_t1, best_t2, color='blue', marker='*', s=200,
            label=f'Optimum: ({best_t1:.1f}, {best_t2:.1f})', zorder=5)
ax1.scatter(1.0, 1.0, color='green', marker='o', s=100,
            label='Uniform (1, 1) = aggregation', zorder=5)
ax1.set_xlabel('θ₁ (controls oldest lag weight)')
ax1.set_ylabel('θ₂ (controls recency)')
ax1.set_title('R² Contour Map')
ax1.legend(fontsize=8)

# Weight profiles for selected parameter combinations
ax2 = axes[1]
lags = np.arange(K)

configs = [
    (1.0, 1.0, 'Uniform (aggregation)', 'gray', '--'),
    (best_t1, best_t2, f'Grid optimum ({best_t1:.1f}, {best_t2:.1f})', 'red', '-'),
    (1.0, 5.0, 'Strong decline (1.0, 5.0)', 'steelblue', '-'),
    (2.0, 2.0, 'Bell-shaped (2.0, 2.0)', 'green', '-'),
]

for t1, t2, label, color, ls in configs:
    w = beta_weights(K, t1, t2)
    r2 = midas_r2(Y, X, t1, t2)
    ax2.plot(lags, w, ls, color=color, linewidth=2, markersize=5,
             label=f'{label} | R²={r2:.3f}')

ax2.axhline(1/K, color='lightgray', linestyle=':', linewidth=1)
for qb in [3, 6]:
    ax2.axvline(qb - 0.5, color='lightgray', linestyle=':', linewidth=1)
ax2.set_xlabel('Lag j (0 = most recent)')
ax2.set_ylabel('Weight')
ax2.set_title('Weight Profiles and their R² Values')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('r2_surface.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Cross-Sections of the R² Surface

Fixing one theta and varying the other shows how sensitive R² is to each parameter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('R² Cross-Sections: Sensitivity to Each Parameter', fontsize=12)

# Fix theta2 at optimal, vary theta1
ax1 = axes[0]
t1_range = np.linspace(0.2, 6.0, 50)
r2_vary_t1 = [midas_r2(Y, X, t1, best_t2) for t1 in t1_range]

ax1.plot(t1_range, r2_vary_t1, 'steelblue', linewidth=2.5)
ax1.axvline(best_t1, color='red', linestyle='--', label=f'Optimal θ₁={best_t1:.2f}')
ax1.axvline(1.0, color='gray', linestyle=':', label='θ₁=1 (uniform if θ₂=1)')
ax1.set_xlabel('θ₁')
ax1.set_ylabel('R²')
ax1.set_title(f'Vary θ₁ (θ₂={best_t2:.1f} fixed)')
ax1.legend(fontsize=9)
ax1.set_ylim(0, None)

# Fix theta1 at optimal, vary theta2
ax2 = axes[1]
t2_range = np.linspace(0.2, 10.0, 50)
r2_vary_t2 = [midas_r2(Y, X, best_t1, t2) for t2 in t2_range]

ax2.plot(t2_range, r2_vary_t2, 'coral', linewidth=2.5)
ax2.axvline(best_t2, color='red', linestyle='--', label=f'Optimal θ₂={best_t2:.2f}')
ax2.axvline(1.0, color='gray', linestyle=':', label='θ₂=1')
ax2.set_xlabel('θ₂')
ax2.set_ylabel('R²')
ax2.set_title(f'Vary θ₂ (θ₁={best_t1:.1f} fixed)')
ax2.legend(fontsize=9)
ax2.set_ylim(0, None)

plt.tight_layout()
plt.savefig('r2_cross_sections.png', dpi=120, bbox_inches='tight')
plt.show()

# Assess smoothness
r2_range_t1 = max(r2_vary_t1) - min(r2_vary_t1)
r2_range_t2 = max(r2_vary_t2) - min(r2_vary_t2)
print(f"R² range when varying θ₁: {r2_range_t1:.4f}")
print(f"R² range when varying θ₂: {r2_range_t2:.4f}")
print(f"\nInterprétation:")
print(f"  {'θ₁ is relatively insensitive' if r2_range_t1 < 0.02 else 'θ₁ has significant impact on R²'}")
print(f"  {'θ₂ is relatively insensitive' if r2_range_t2 < 0.02 else 'θ₂ has significant impact on R²'}")
print(f"\nIf R² range is small, the model is robust to the weight shape — OLS aggregate may be adequate.")

## 4. Animated Weight Evolution: How Weights Change as $\theta_2$ Increases

As $\theta_2$ increases from 1 to 10, the weight function transitions from uniform to strongly front-loaded. This visualization shows the continuous nature of the parameter space.

In [ ]:
# Static multi-panel showing weight evolution
theta2_values = [1.0, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]
theta1_fixed = 1.0

fig, axes = plt.subplots(1, len(theta2_values), figsize=(16, 3.5))
fig.suptitle(f'Weight Evolution as θ₂ Increases (θ₁={theta1_fixed} fixed, K={K})',
             fontsize=11, fontweight='bold')

lags = np.arange(K)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(theta2_values)))

for ax, t2, color in zip(axes, theta2_values, colors):
    w = beta_weights(K, theta1_fixed, t2)
    r2 = midas_r2(Y, X, theta1_fixed, t2)
    
    ax.bar(lags, w, color=color, alpha=0.85)
    ax.axhline(1/K, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_title(f'θ₂={t2:.0f}\nR²={r2:.3f}', fontsize=9)
    ax.set_ylim(0, max(beta_weights(K, theta1_fixed, 10.0)) + 0.02)
    ax.set_xticks([])
    if axes.tolist().index(ax) == 0:
        ax.set_ylabel('Weight')
    
    # Mark current-quarter boundary
    ax.axvline(2.5, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)

# Add arrows between panels
plt.tight_layout()
plt.savefig('weight_evolution.png', dpi=120, bbox_inches='tight')
plt.show()

print("Observation: As θ₂ increases:")
print(f"  - Weight on lag 0 (most recent): ", end="")
for t2 in [1.0, 3.0, 5.0, 10.0]:
    w = beta_weights(K, 1.0, t2)
    print(f"{w[0]:.3f} (θ₂={t2:.0f}) ", end="")
print()
print(f"  - R²: ", end="")
for t2 in [1.0, 3.0, 5.0, 10.0]:
    r2 = midas_r2(Y, X, 1.0, t2)
    print(f"{r2:.4f} (θ₂={t2:.0f}) ", end="")
print()

## 5. Quarterly Weight Summary

Group the K monthly weights by quarter to get a cleaner economic interpretation: how much of the total predictive power comes from the current quarter vs. previous quarters?

In [ ]:
# Compare quarterly weight allocation across parameter values
theta_specs = [
    (1.0, 1.0, 'Uniform (β₁=1, θ₂=1)'),
    (best_t1, best_t2, f'Estimated optimum'),
    (1.0, 5.0, 'Strong decline (θ₂=5)'),
    (2.0, 2.0, 'Bell-shaped (2, 2)'),
]

n_quarters = K // 3
quarter_names = ['Current Q', 'Q-1', 'Q-2'] + [f'Q-{i}' for i in range(3, n_quarters)]

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Weight Distribution by Quarter\n(How much does each quarter contribute?)',
             fontsize=12, fontweight='bold')

x_pos = np.arange(n_quarters)
width = 0.18
colors_list = ['steelblue', 'coral', 'green', 'purple']

for i, (t1, t2, label) in enumerate(theta_specs):
    w = beta_weights(K, t1, t2)
    # Sum weights within each quarter (groups of 3 consecutive lags)
    q_weights = [w[q*3:(q+1)*3].sum() for q in range(n_quarters)]
    r2 = midas_r2(Y, X, t1, t2)
    ax.bar(x_pos + i*width, q_weights, width,
           color=colors_list[i], alpha=0.85,
           label=f'{label} | R²={r2:.3f}')

ax.axhline(1/n_quarters, color='gray', linestyle='--', linewidth=1,
           label=f'Equal quarter allocation (1/{n_quarters}={1/n_quarters:.2f})')
ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels(quarter_names[:n_quarters])
ax.set_ylabel('Total weight for quarter')
ax.set_title('Weight Allocation by Quarterly Group')
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('quarterly_weights.png', dpi=120, bbox_inches='tight')
plt.show()

# Print the table
print("\nQuarterly Weight Allocation Table:")
print(f"{'Model':<35} " + " ".join([f"{q:>9s}" for q in quarter_names[:n_quarters]]))
print("-" * (35 + 10 * n_quarters))

for t1, t2, label in theta_specs:
    w = beta_weights(K, t1, t2)
    q_weights = [w[q*3:(q+1)*3].sum() for q in range(n_quarters)]
    row = f"{label:<35} " + " ".join([f"{qw:>9.1%}" for qw in q_weights])
    print(row)

## Summary

### What You Visualized
1. The **R² surface** over $(\theta_1, \theta_2)$ space — shows the optimization landscape is smooth and unimodal (good for NLS convergence)
2. **Cross-sections** of R² — shows which parameter has more impact on fit
3. **Weight evolution** as $\theta_2$ increases — shows the continuous transition from uniform to front-loaded
4. **Quarterly weight allocation** — shows how different weight shapes distribute influence across quarterly periods

### Key Insights
- The R² surface is generally smooth — NLS converges reliably for Beta polynomial
- $\theta_2$ (recency parameter) typically has more impact on R² than $\theta_1$
- The estimated optimal weights concentrate on the current quarter (~50-60% typical)
- Very high $\theta_2$ values (strongly front-loaded) don't always give the best fit — the model can overweight recent months

### What's Next
Module 02 covers the formal estimation framework: nonlinear least squares, HAC standard errors, and model selection criteria. We'll learn how to quantify uncertainty around the estimated weight parameters.

---
*Next: Module 01 Exercise, then Module 02 — Estimation and Inference*